In [ ]:
import sisl as si
import numpy as np
import matplotlib.pyplot as plt
import warnings

assert si.__version_tuple__ > (
    0,
    20,
    0,
), "Requires sisl version > 0.20.0, please wait for the next release, or use the developer version"

warnings.filterwarnings("ignore", "InterpolatedUnivariateSpline")
%matplotlib inline

## Slater Koster models

The basis of doing tight-binding calculations are typically grounded in the choices of parameters for couplings and on-site values.

However, that can be rather tedious to discover. In this advanced tutorial we will guide you through how to create a Slater Koster models using the DFTB+ model parameters, and extracting the parameters from a Hamiltonian.

## Using DFTB+ values

If you are unaware of the DFTB+ code, have a look [here](https://dftb.org). It's extremely fast, yet retains a high degree of precision.

DFTB+ shares the model parameters from a set of SK model files which contains interpolation values for range parameters of coupling elements and on-site values.

In this folder, we've shared the `mio` set which contains certain species parameterizations.

### Citations

The `mio` dataset we'll be using in this tutorial is work accummulated by these publications:

- [PRB98]  	Phys. Rev. B 58 (1998) 7260
- [JMS01]  	J. Mol. Struc. (THEOCHEM) 541 (2001) 185
- [JCP05] 	J. Chem. Phys 122 (122) 114110 
- [JCTC06] 	J. Chem. Theor. and Comput. 2006
- [JCP01] 	J. Chem. Phys. 114 (2001) 5149
- [CP01] 		Chem. Phys. 263 (2001) 203
- [PSS00] 	Phys. Stat. Sol. b 217 (2000) 357. 	 
- [JCTC2011]      Gaus,Cui,Elstner,J.Chem.TheoryComput.2011,7,931-948.
- [JCTC2008]      Yang,Yu,York,Elstner,Cui,J.Chem.TheoryComput.2008,4,2067-2084.

### Extracting the SK model parameters

DFTB+ has their own tables and we are now reading in these tables, and converting them to Slater-Koster models in `sisl`.

**NOTE**: Using Slater-Koster models is not simple, it's not a simple choose and pick and merge different sets together. One has to be careful when choosing sets and when to mix them. This is as delicate as pseudo-potentials.

In [ ]:
from itertools import combinations_with_replacement


def get_mio(s1, s2):
    return si.get_sile(f"mio-1-1/{s1}-{s2}.skf").read_slater_koster()


species = ["O", "C", "H"]

sk = si.SlaterKoster.merge(
    *(get_mio(s1, s2) for s1, s2 in combinations_with_replacement(species, 2))
)
# Lets see the atoms associated with this object
print(sk.atoms)

Now we have a Slater-Koster model which has all combinatorial parameters of the above species.

This concludes the ingredients for constructing a large variety of Hamiltonians using Carbon, Oxygen and Hydrogen. Here, we do not worry whether the physics is correct (the `mio` set might not be the best for GNR with Oxygen ad-atoms).

We'll continue anyways, and create a system of GNR with Hydrogen ends.

In [ ]:
gnr = si.geom.zgnr(7, atoms=sk.atoms["C"])

# Now saturate with Hydrogens
from sisl.geom import Saturate, saturate, AtomCategory

edge = AtomCategory(neighbors=dict(max=2, R=1.5))
gnr_H = saturate(
    gnr,
    [
        # Add H atoms with a bond-length of 1.09
        (
            edge & AtomCategory(Z="C"),
            Saturate(sk.atoms["H"], bond=1.09, coordination=3, R=1.5),
        ),
    ],
)
gnr_H.plot(axes="-yx")

Now we have a fully functional Z-GNR with Hydrogen saturations. Lets do the Hamiltonian and do a band-structure.

In [ ]:
H = si.Hamiltonian(gnr_H, nnzpr=50, orthogonal=False)
H.construct(sk)

In [ ]:
band = si.BandStructure(
    H,
    [[0.0, 0.0, 0], [0.5, 0.0, 0.0]],
    101,
    [r"$\Gamma$", "K"],
)
eigs = band.apply.array.eigh()

In [ ]:
# Retrieve the tick-marks and the linear k points
xtick, xtick_label = band.lineartick()
lk = band.lineark()
plt.plot(lk, eigs)
plt.ylim(-15, 5)

plt.ylabel("Eigenspectrum [eV]")
plt.gca().xaxis.set_ticks(xtick)
plt.gca().set_xticklabels(xtick_label)

# Also plot x-major lines at the ticks
ymin, ymax = plt.gca().get_ylim()
for tick in xtick:
    plt.plot([tick, tick], [ymin, ymax], "k")

Now we've shown we can do Slater Koster models for a system like this.

We can go even further and add an Oxygen atom.

First tile the geometry a number of times and place the Oxygen atom at the middle above the atom.

In [ ]:
gnr_long = gnr_H.tile(7, 0)
gnr_long.plot(axes="-yx")

In [ ]:
gnr_long_O = saturate(
    gnr_long,
    [
        # Add O atoms with a bond-length resulting from the empirical radii of the respective atoms
        (
            gnr_long.close(gnr_long.center(), R=1)[0],
            Saturate(sk.atoms["O"], coordination=4, R=1.5),
        ),
    ],
)
gnr_long_O.plot(axes="-yx")

In [ ]:
H_O = si.Hamiltonian(gnr_long_O, orthogonal=False, nnzpr=50)
H_O.construct(sk)

We can then store the resulting matrices and perform TBtrans transport calculations using these custom models (entirely built using SK models).

In [ ]:
H.tile(2, 0).write("ELEC.nc")
H_O.write("DEVICE.nc")

Now you can do TBtrans calculations and examine the transport properties and density of states. The `RUN.fdf` file is already prepared, so you won't have to edit it (unless you want more data calculated).  
Since this model has quite a bit of orbitals, it will take more time to compute.

## Creating Slater Koster models from a Hamiltonian

There are numerous ways to backtrack a Hamiltonian object into a Slater Koster model. This is by no means an easy task, and it would require extensive testing for the various environments the atoms exists in.  
So be *warned*, that the following example is not a *black-box* solution. It requires miticulous understanding and testing!

In this example it goes well because the environment the atoms sees is consistent, and with no degrees of freedom.  

The Slater Koster object can readily extract the parameters from a Hamiltonian by doing the operations as outlined in [McMahan][8].
You can construct the Slater Koster by simply passing the Hamiltonian. There are some requirements for this Hamiltonian, e.g. it needs to know which orbitals are $s$, $p$ or $d$ orbitals. So if you are reading Siesta Hamiltonians, you would need to ensure the orbitals and atoms are correctly understood.

[8]: https://www.doi.org/10.1103/PhysRevB.56.12250 "A. K. McMahan and J. E. Klepeis"

In [ ]:
sk_new = si.SlaterKoster.from_hamiltonian(H_O)

In [ ]:
H_new = H_O.copy()
H_new.reset()
print("nnz at init = , H_new.nnz) # check the print-out says it has 0 nnz
H_new.construct(sk_new)
dH = H_O - H_new
dH.eliminate_zeros(1e-10)
print("nnz comparison =", dH.nnz)

## Exercises

- Instead of doing a Z-GNR with Hydrogens saturating the Carbons, try and use Oxygen as the saturating atom.  
  Then re-do the bandstructure and check for differences.
- Now do the GNR but with Hydrogens saturating the top-edge, and Oxygen the bottom-edge.  
  [This page](https://zerothi.github.io/sisl/tutorials/tutorial_categories.html#5.-The-position-of-atoms) may come in handy to select atoms based on their coordinates.